05_export_tables.py

Tugas Orang 5 — Ekspor tabel hasil akhir untuk artikel IEEE dan laporan.

Input:
- data/interim/fcm_membership_geojson.csv  (output script 01)
- data/processed/cluster_centroids_standardized.csv
- data/processed/external_validation.csv
- data/processed/cluster_profiles.csv

Output:
- outputs/tables/tabel_klaster_provinsi.csv
    → daftar lengkap 36 provinsi + klaster + membership + status
- outputs/tables/tabel_centroid_klaster.csv
    → nilai z-score centroid per klaster (siap masuk tabel artikel)
- outputs/tables/tabel_validasi_eksternal.csv
    → ringkasan prevalensi stunting per klaster
- outputs/tables/tabel_ringkasan_klaster.csv
    → profil klaster: skor risiko, jumlah provinsi, mean membership

In [17]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT   = Path().resolve().parent
HARMONIZED     = PROJECT_ROOT / "data" / "interim" / "fcm_membership_geojson.csv"
CENTROIDS      = PROJECT_ROOT / "outputs" / "model" / "cluster_centroids_standardized.csv"
EXT_VALID      = PROJECT_ROOT / "outputs" / "analysis" / "external_validation.csv"
PROFILES       = PROJECT_ROOT / "outputs" / "analysis" / "cluster_profiles.csv"
TABLE_DIR      = PROJECT_ROOT / "outputs" / "tables"

VARIABLE_LABELS = {
    "maternal_age_risk_z":      "Risiko Usia Ibu Hamil (z)",
    "low_knowledge_z":          "Rendahnya Pengetahuan Stunting (z)",
    "water_no_or_unimproved_z": "Air Minum Tidak Layak/Tanpa Akses (z)",
    "water_limited_z":          "Air Minum Akses Terbatas (z)",
    "sanitation_babs_z":        "Sanitasi BABS (z)",
    "sanitation_unimproved_z":  "Sanitasi Tidak Layak (z)",
}

In [18]:
def export_cluster_province_table():
    df = pd.read_csv(HARMONIZED)

    out = df[[
        "province_name", "crisp_cluster",
        "membership_cluster_1", "membership_cluster_2",
        "maximum_membership", "membership_margin",
        "membership_status", "stunting_prevalence_pct", "cluster_label",
    ]].copy()

    out = out.rename(columns={
        "province_name":         "Nama Provinsi (FCM)",
        "crisp_cluster":         "Klaster",
        "membership_cluster_1":  "Membership Klaster 1",
        "membership_cluster_2":  "Membership Klaster 2",
        "maximum_membership":    "Membership Maksimum",
        "membership_margin":     "Margin Membership",
        "membership_status":     "Status Membership",
        "stunting_prevalence_pct": "Prevalensi Stunting (%)",
        "cluster_label":         "Label Klaster",
    })

    # Urutkan: klaster asc, membership desc
    out = out.sort_values(["Klaster", "Membership Maksimum"], ascending=[True, False])
    out = out.reset_index(drop=True)

    # Round float kolom
    for col in ["Membership Klaster 1", "Membership Klaster 2",
                "Membership Maksimum", "Margin Membership"]:
        out[col] = out[col].round(4)

    out_path = TABLE_DIR / "tabel_klaster_provinsi.csv"
    out.to_csv(out_path, index=False)
    print(f"Tabel klaster provinsi disimpan → {out_path}  ({len(out)} baris)")
    return out

In [19]:
def export_centroid_table():
    df = pd.read_csv(CENTROIDS)
    df = df.set_index("cluster")

    cols = [c for c in VARIABLE_LABELS if c in df.columns]
    df   = df[cols].rename(columns=VARIABLE_LABELS)
    df.index = ["Klaster 1 (Risiko Rendah)", "Klaster 2 (Risiko Tinggi)"]
    df = df.round(4)

    out_path = TABLE_DIR / "tabel_centroid_klaster.csv"
    df.to_csv(out_path)
    print(f"Tabel centroid disimpan       → {out_path}")
    return df

In [20]:
def export_external_validation():
    df = pd.read_csv(EXT_VALID)
    df = df.rename(columns={
        "cluster":                  "Klaster",
        "cluster_label":            "Label Klaster",
        "number_of_provinces":      "Jumlah Provinsi",
        "mean_stunting_prevalence": "Rata-rata Prevalensi Stunting (%)",
        "median_stunting_prevalence": "Median (%)",
        "minimum_stunting_prevalence": "Min (%)",
        "maximum_stunting_prevalence": "Maks (%)",
        "standard_deviation":       "Std. Deviasi",
    })
    for col in ["Rata-rata Prevalensi Stunting (%)", "Median (%)",
                "Min (%)", "Maks (%)", "Std. Deviasi"]:
        df[col] = df[col].round(2)

    out_path = TABLE_DIR / "tabel_validasi_eksternal.csv"
    df.to_csv(out_path, index=False)
    print(f"Tabel validasi eksternal      → {out_path}")
    return df

In [21]:
def export_cluster_profiles():
    df = pd.read_csv(PROFILES)
    df = df.rename(columns={
        "cluster":                "Klaster",
        "family_score":           "Skor Faktor Keluarga",
        "water_score":            "Skor Faktor Air",
        "sanitation_score":       "Skor Faktor Sanitasi",
        "overall_risk_score":     "Skor Risiko Keseluruhan",
        "dominant_dimension":     "Dimensi Dominan",
        "dominant_indicator":     "Indikator Dominan",
        "cluster_label":          "Label Klaster",
        "number_of_provinces":    "Jumlah Provinsi",
        "mean_maximum_membership":"Rata-rata Max. Membership",
    })
    for col in ["Skor Faktor Keluarga", "Skor Faktor Air", "Skor Faktor Sanitasi",
                "Skor Risiko Keseluruhan", "Rata-rata Max. Membership"]:
        df[col] = df[col].round(4)

    out_path = TABLE_DIR / "tabel_profil_klaster.csv"
    df.to_csv(out_path, index=False)
    print(f"Tabel profil klaster          → {out_path}")
    return df

In [22]:
def print_summary(df_prov, df_valid):
    print("\n" + "=" * 60)
    print("RINGKASAN HASIL CLUSTERING FCM")
    print("=" * 60)

    for c in sorted(df_prov["Klaster"].dropna().unique()):
        sub = df_prov[df_prov["Klaster"] == c]
        label = sub["Label Klaster"].iloc[0]
        n = len(sub)
        mean_prev = df_valid.loc[
            df_valid["Klaster"] == c, "Rata-rata Prevalensi Stunting (%)"
        ].values
        prev_str = f"{mean_prev[0]:.1f}%" if len(mean_prev) else "N/A"
        print(f"\nKlaster {int(c)}: {label}")
        print(f"  Jumlah provinsi          : {n}")
        print(f"  Mean prevalensi stunting : {prev_str}")
        ambig = sub[sub["Status Membership"] == "Ambigu tinggi"]["Nama Provinsi (FCM)"].tolist()
        if ambig:
            print(f"  Provinsi ambigu tinggi   : {', '.join(ambig)}")

    print("\n" + "=" * 60)

In [23]:
def main():
    TABLE_DIR.mkdir(parents=True, exist_ok=True)
    df_prov  = export_cluster_province_table()
    export_centroid_table()
    df_valid = export_external_validation()
    export_cluster_profiles()
    print_summary(df_prov, df_valid)

### Jalankan Pipeline

In [24]:
if __name__ == "__main__":
    main()

Tabel klaster provinsi disimpan → C:\muti\SMT 6\PKK_PROJECT AKHIR\pkk-stunting-fcm-clustering\outputs\tables\tabel_klaster_provinsi.csv  (36 baris)
Tabel centroid disimpan       → C:\muti\SMT 6\PKK_PROJECT AKHIR\pkk-stunting-fcm-clustering\outputs\tables\tabel_centroid_klaster.csv
Tabel validasi eksternal      → C:\muti\SMT 6\PKK_PROJECT AKHIR\pkk-stunting-fcm-clustering\outputs\tables\tabel_validasi_eksternal.csv
Tabel profil klaster          → C:\muti\SMT 6\PKK_PROJECT AKHIR\pkk-stunting-fcm-clustering\outputs\tables\tabel_profil_klaster.csv

RINGKASAN HASIL CLUSTERING FCM

Klaster 1: Profil faktor risiko relatif lebih rendah
  Jumlah provinsi          : 23
  Mean prevalensi stunting : 19.6%
  Provinsi ambigu tinggi   : PAPUA, SULAWESI SELATAN

Klaster 2: Profil faktor risiko relatif lebih tinggi
  Jumlah provinsi          : 13
  Mean prevalensi stunting : 27.0%
  Provinsi ambigu tinggi   : SULAWESI UTARA

